## ISIC 2018 End-to-End Pipeline: Lesion Segmentation + Disease Classification

Chains Mask R-CNN (Task 1 segmentation) with EfficientNet-B0 (Task 3
classification) into a single inference pipeline.
For each image:
   1. Mask R-CNN localises the lesion and produces a binary mask.
   2. The predicted bounding box crops the lesion region.
   3. EfficientNet-B0 classifies the crop into one of 7 HAM10000 disease classes.

**Joint evaluation on 50 validation images**:
   - Mean thresholded Jaccard (T=0.65): segmentation quality
   - Balanced accuracy: classification quality over detected images
   - Combined score: arithmetic mean of the two

**Datasets required**:
   - tschandl/isic2018-challenge-task1-data-segmentation (images + masks)
   - kmader/skin-cancer-mnist-ham10000 (metadata CSV for class labels)

**Checkpoints required (add as Kaggle dataset inputs)**:
   - best_maskrcnn_isic2018.pth (Task 1 notebook output)
   - best_efficientnet_b0_run5_ham10000.pth (Task 3 notebook output)

Reference: Codella et al., arXiv:1902.03368, 2019.

## 1. Imports and configuration

In [ ]:
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data
import torchvision.transforms.v2.functional as TF
import timm
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from PIL import Image
from sklearn.metrics import balanced_accuracy_score
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# Paths -- update if your dataset input names differ
# ---------------------------------------------------------------------------
TASK1_BASE = Path("/kaggle/input/isic2018-challenge-task1-data-segmentation")
TASK3_BASE = Path("/kaggle/input/skin-cancer-mnist-ham10000")
CKPT_SEG = Path("/kaggle/input/isic-checkpoints/best_maskrcnn_isic2018.pth")
CKPT_CLS = Path("/kaggle/input/isic-checkpoints/best_efficientnet_b0_run5_ham10000.pth")

CFG = {
    "img_dir": TASK1_BASE / "ISIC2018_Task1-2_Training_Input",
    "mask_dir": TASK1_BASE / "ISIC2018_Task1_Training_GroundTruth",
    "metadata": TASK3_BASE / "HAM10000_metadata.csv",
    "img_size": 512,
    "cls_img_size": 224,
    "val_fraction": 0.2,
    "seed": 42,
    "n_eval": 50,  # number of validation images to evaluate
    "n_visualise": 4,  # number of pipeline outputs to visualise
    "num_classes_seg": 2,  # background + lesion
    "num_classes_cls": 7,
    "score_threshold": 0.5,
    "mask_threshold": 0.5,
    "jaccard_threshold": 0.65,
    "cls_mean": (0.485, 0.456, 0.406),
    "cls_std": (0.229, 0.224, 0.225),
}

CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
torch.cuda.manual_seed_all(CFG["seed"])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## 2. Combined dataset

In [ ]:
class ISICPipelineDataset(torch.utils.data.Dataset):
    """
    Combined ISIC 2018 dataset for end-to-end pipeline evaluation.

    Yields (image, gt_mask, gt_class_label) for images that appear in both
    the Task 1 segmentation dataset and the HAM10000 metadata. Images with
    no metadata match are excluded at construction time.

    Images are resized to img_size x img_size. No normalisation is applied
    to the image tensor -- Mask R-CNN expects [0, 1] float input without
    ImageNet normalisation.

    Args:
        img_dir:   Directory containing ISIC_XXXXXXX.jpg files.
        mask_dir:  Directory containing ISIC_XXXXXXX_segmentation.png files.
        metadata:  Path to HAM10000_metadata.csv.
        img_ids:   List of ISIC image IDs to include before metadata filtering.
        img_size:  Both spatial dimensions are resized to this value.
    """

    def __init__(
            self,
            img_dir: Path,
            mask_dir: Path,
            metadata: Path,
            img_ids: list[str],
            img_size: int = 512,
    ) -> None:
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_size = img_size

        meta = pd.read_csv(metadata)[["image_id", "dx"]].drop_duplicates("image_id")
        meta_index = dict(zip(meta["image_id"], meta["dx"]))

        self.records: list[tuple[str, int]] = []
        for img_id in img_ids:
            dx = meta_index.get(img_id)
            if dx is not None and dx in CLASS_TO_IDX:
                self.records.append((img_id, CLASS_TO_IDX[dx]))

        print(
            f"Pipeline dataset: {len(self.records)} images with matched class labels "
            f"(excluded {len(img_ids) - len(self.records)} without metadata match)"
        )

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, int]:
        img_id, class_label = self.records[idx]

        img = Image.open(self.img_dir / f"{img_id}.jpg").convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.Resampling.BILINEAR)
        img_tensor = TF.to_dtype(TF.to_image(img), dtype=torch.float32, scale=True)

        mask = Image.open(self.mask_dir / f"{img_id}_segmentation.png").convert("L")
        mask = mask.resize((self.img_size, self.img_size), Image.Resampling.NEAREST)
        mask_tensor = torch.from_numpy((np.array(mask) > 127).astype(np.uint8))

        return img_tensor, mask_tensor, class_label


def build_pipeline_dataset(cfg: dict) -> ISICPipelineDataset:
    """
    Build the pipeline evaluation dataset using the same 80/20 random split
    as the Task 1 segmentation notebook, then take the first n_eval images
    from the validation split.

    The split uses the same seed and shuffle logic as Task 1 to ensure the
    pipeline is evaluated on held-out images the segmentation model has not
    seen during training.
    """
    all_ids = sorted(p.stem for p in cfg["img_dir"].glob("*.jpg"))
    rng = random.Random(cfg["seed"])
    rng.shuffle(all_ids)
    split = int(len(all_ids) * (1 - cfg["val_fraction"]))
    val_ids = all_ids[split:][:cfg["n_eval"]]

    return ISICPipelineDataset(
        img_dir=cfg["img_dir"],
        mask_dir=cfg["mask_dir"],
        metadata=cfg["metadata"],
        img_ids=val_ids,
        img_size=cfg["img_size"],
    )


pipeline_dataset = build_pipeline_dataset(CFG)
pipeline_loader = torch.utils.data.DataLoader(
    pipeline_dataset, batch_size=1, shuffle=False, num_workers=2,
)
print(f"Evaluation loader: {len(pipeline_dataset)} images")

## 3. Model definitions

In [ ]:
def build_seg_model(num_classes: int) -> nn.Module:
    """
    Load Mask R-CNN (ResNet-50-FPN) pretrained on COCO and replace heads
    for num_classes output classes. Mirrors Task 1 notebook exactly.
    """
    model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    in_features_box = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features_box, num_classes)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_dim = model.roi_heads.mask_predictor.conv5_mask.out_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_dim, num_classes)
    return model


def build_cls_model(num_classes: int) -> nn.Module:
    """
    Load EfficientNet-B0 pretrained on ImageNet with classification head
    replaced for num_classes output classes. Mirrors Task 3 notebook exactly.
    """
    return timm.create_model(
        "efficientnet_b0", pretrained=False, num_classes=num_classes, drop_rate=0.4,
    )


seg_model = build_seg_model(CFG["num_classes_seg"]).to(DEVICE)
cls_model = build_cls_model(CFG["num_classes_cls"]).to(DEVICE)

seg_ckpt = torch.load(CKPT_SEG, map_location=DEVICE)
cls_ckpt = torch.load(CKPT_CLS, map_location=DEVICE)

seg_model.load_state_dict(seg_ckpt["model_state_dict"])
cls_model.load_state_dict(cls_ckpt["model_state_dict"])

seg_model.eval()
cls_model.eval()

print(f"Segmentation checkpoint: epoch {seg_ckpt['epoch']}, val_jaccard={seg_ckpt['val_jaccard']:.4f}")
print(f"Classification checkpoint: epoch {cls_ckpt['epoch']}, val_bal_acc={cls_ckpt['val_bal_acc']:.4f}")

## 4. Pipeline definition

In [ ]:
@dataclass(frozen=True)
class PredictionResult:
    """
    Output of ISICPipeline.predict() for a single image.

    Attributes:
        mask:                Binary predicted lesion mask, shape (H, W), dtype bool.
                             All-zero if no detection passed the score threshold.
        class_label:         Predicted class index in [0, 6], or -1 if detection failed.
        class_probabilities: Softmax probabilities, shape (7,), dtype float32.
                             None if detection failed.
        detection_failed:    True if Mask R-CNN produced no detection above
                             score_threshold.
        score:               Mask R-CNN confidence score. 0.0 if detection failed.
        crop:                Unnormalised crop tensor (3, cls_img_size, cls_img_size)
                             in [0, 1] for visualisation. None if detection failed.
    """
    mask: np.ndarray
    class_label: int
    class_probabilities: np.ndarray | None
    detection_failed: bool
    score: float
    crop: torch.Tensor | None


class ISICPipeline:
    """
    End-to-end ISIC lesion analysis pipeline.

    See isic_pipeline.py in the local repository for full documentation.
    This version adds a `crop` field to PredictionResult for visualisation.
    """

    def __init__(
            self,
            seg_model: nn.Module,
            cls_model: nn.Module,
            img_size: int = 512,
            cls_img_size: int = 224,
            score_threshold: float = 0.5,
            mask_threshold: float = 0.5,
            jaccard_threshold: float = 0.65,
            cls_mean: tuple[float, float, float] = (0.485, 0.456, 0.406),
            cls_std: tuple[float, float, float] = (0.229, 0.224, 0.225),
    ) -> None:
        self.seg_model = seg_model.eval()
        self.cls_model = cls_model.eval()
        self.img_size = img_size
        self.cls_img_size = cls_img_size
        self.score_threshold = score_threshold
        self.mask_threshold = mask_threshold
        self.jaccard_threshold = jaccard_threshold
        self.cls_mean = cls_mean
        self.cls_std = cls_std
        self.device = next(seg_model.parameters()).device

    def _run_segmentation(
            self, image: torch.Tensor
    ) -> tuple[np.ndarray, np.ndarray | None, float]:
        H, W = image.shape[1], image.shape[2]
        output = self.seg_model([image])[0]
        keep = output["scores"] >= self.score_threshold
        if not keep.any():
            return np.zeros((H, W), dtype=bool), None, 0.0
        best_idx = output["scores"][keep].argmax()
        score = output["scores"][keep][best_idx].item()
        soft_mask = output["masks"][keep][best_idx, 0].cpu().numpy()
        binary_mask = soft_mask >= self.mask_threshold
        box = output["boxes"][keep][best_idx].cpu().numpy()
        return binary_mask, box, score

    def _crop_and_preprocess(
            self, image: torch.Tensor, box: np.ndarray
    ) -> tuple[torch.Tensor, torch.Tensor] | tuple[None, None]:
        """
        Returns (normalised_crop, display_crop) or (None, None) if degenerate.
        display_crop is unnormalised [0, 1] for visualisation.
        """
        H, W = image.shape[1], image.shape[2]
        x1 = max(0, int(box[0]))
        y1 = max(0, int(box[1]))
        x2 = min(W, int(np.ceil(box[2])))
        y2 = min(H, int(np.ceil(box[3])))
        if x2 <= x1 or y2 <= y1:
            return None, None
        crop = image[:, y1:y2, x1:x2]
        crop = F.interpolate(
            crop.unsqueeze(0),
            size=(self.cls_img_size, self.cls_img_size),
            mode="bilinear",
            align_corners=False,
        )
        display_crop = crop[0].cpu()
        mean = torch.tensor(self.cls_mean, device=self.device).view(1, 3, 1, 1)
        std = torch.tensor(self.cls_std, device=self.device).view(1, 3, 1, 1)
        return (crop - mean) / std, display_crop

    @staticmethod
    def _compute_jaccard(pred: np.ndarray, gt: np.ndarray, threshold: float) -> float:
        pred = pred.astype(bool)
        gt = gt.astype(bool)
        union = (pred | gt).sum()
        if union == 0:
            return 0.0
        iou = float((pred & gt).sum() / union)
        return iou if iou >= threshold else 0.0

    @torch.no_grad()
    def predict(self, image: torch.Tensor) -> PredictionResult:
        """
        Run the full pipeline on a single image.

        Args:
            image: FloatTensor (3, H, W) in [0, 1], no ImageNet normalisation.

        Returns:
            PredictionResult.
        """
        image = image.to(self.device)
        binary_mask, box, score = self._run_segmentation(image)

        if box is None:
            return PredictionResult(
                mask=binary_mask, class_label=-1, class_probabilities=None,
                detection_failed=True, score=0.0, crop=None,
            )

        norm_crop, display_crop = self._crop_and_preprocess(image, box)
        if norm_crop is None:
            return PredictionResult(
                mask=binary_mask, class_label=-1, class_probabilities=None,
                detection_failed=True, score=score, crop=None,
            )

        logits = self.cls_model(norm_crop)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
        label = int(probs.argmax())

        return PredictionResult(
            mask=binary_mask, class_label=label, class_probabilities=probs,
            detection_failed=False, score=score, crop=display_crop,
        )

    @torch.no_grad()
    def evaluate(
            self, dataloader: torch.utils.data.DataLoader
    ) -> tuple[float, float | None, float | None, int, int]:
        """
        Evaluate the pipeline on a DataLoader yielding (image, gt_mask, gt_label).

        Returns:
            Tuple of (mean_jaccard, balanced_accuracy, combined_score,
                      n_total, n_detection_failed).
        """
        self.seg_model.eval()
        self.cls_model.eval()

        jaccard_scores: list[float] = []
        cls_preds: list[int] = []
        cls_labels: list[int] = []
        n_detection_failed = 0

        for image, gt_mask, gt_label in dataloader:
            if image.shape[0] != 1:
                raise ValueError(
                    f"evaluate() requires batch_size=1, got {image.shape[0]}."
                )
            result = self.predict(image[0])
            gt_mask_np = gt_mask[0].numpy().astype(bool)
            gt_label_int = int(gt_label.item())

            jaccard_scores.append(
                self._compute_jaccard(result.mask, gt_mask_np, self.jaccard_threshold)
            )

            if result.detection_failed:
                n_detection_failed += 1
            else:
                cls_preds.append(result.class_label)
                cls_labels.append(gt_label_int)

        n_total = len(jaccard_scores)
        mean_jaccard = float(np.mean(jaccard_scores)) if jaccard_scores else 0.0

        if cls_preds:
            bal_acc = float(balanced_accuracy_score(cls_labels, cls_preds))
            combined = (mean_jaccard + bal_acc) / 2.0
        else:
            bal_acc = None
            combined = None

        return mean_jaccard, bal_acc, combined, n_total, n_detection_failed

## 5. Instantiate pipeline and run evaluation

In [ ]:
pipeline = ISICPipeline(
    seg_model=seg_model,
    cls_model=cls_model,
    img_size=CFG["img_size"],
    cls_img_size=CFG["cls_img_size"],
    score_threshold=CFG["score_threshold"],
    mask_threshold=CFG["mask_threshold"],
    jaccard_threshold=CFG["jaccard_threshold"],
    cls_mean=CFG["cls_mean"],
    cls_std=CFG["cls_std"],
)

mean_jaccard, bal_acc, combined, n_total, n_failed = pipeline.evaluate(pipeline_loader)

print(f"Evaluation over {n_total} images ({n_failed} detection failures)")
print(f"Mean thresholded Jaccard (T=0.65): {mean_jaccard:.4f}")
print(f"Balanced accuracy (classification): {bal_acc:.4f if bal_acc is not None else 'N/A'}")
print(f"Combined score:                     {combined:.4f if combined is not None else 'N/A'}")
print(f"Detection failure rate:             {n_failed}/{n_total} ({100 * n_failed / n_total:.1f}%)")

## 6. Visualisation: 4 pipeline outputs

In [ ]:
@torch.no_grad()
def collect_pipeline_results(
        pipeline: ISICPipeline,
        dataset: ISICPipelineDataset,
        n: int,
) -> list[dict]:
    """
    Run the pipeline on the first n images of the dataset and collect results
    for visualisation.

    Args:
        pipeline: Instantiated ISICPipeline.
        dataset:  ISICPipelineDataset.
        n:        Number of images to process.

    Returns:
        List of dicts with keys: image, gt_mask, gt_label, result.
    """
    records = []
    for idx in range(min(n, len(dataset))):
        image, gt_mask, gt_label = dataset[idx]
        result = pipeline.predict(image)
        records.append({
            "image": image,
            "gt_mask": gt_mask.numpy().astype(bool),
            "gt_label": gt_label,
            "result": result,
        })
    return records


def visualise_pipeline_outputs(
        records: list[dict],
        n: int = 4,
        save_path: Path | None = None,
) -> None:
    """
    Visualise n pipeline outputs, one per row.

    Columns: input image | predicted mask | lesion crop | class prediction.

    Args:
        records:   Output of collect_pipeline_results.
        n:         Number of rows to display.
        save_path: If provided, save the figure at 150 dpi.
    """
    n = min(n, len(records))
    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    col_titles = ["Input image", "Predicted mask", "Lesion crop", "Prediction"]
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=11, fontweight="bold")

    for row, rec in enumerate(records[:n]):
        image = rec["image"].permute(1, 2, 0).numpy().clip(0, 1)
        gt_label = rec["gt_label"]
        result = rec["result"]

        axes[row, 0].imshow(image)
        axes[row, 0].set_ylabel(
            f"GT: {CLASSES[gt_label]}", fontsize=9, rotation=0,
            labelpad=80, va="center",
        )

        axes[row, 1].imshow(result.mask, cmap="gray", vmin=0, vmax=1)
        axes[row, 1].set_title(
            f"score={result.score:.2f}" if not result.detection_failed else "no detection",
            fontsize=8,
        )

        if result.crop is not None:
            crop_img = result.crop.permute(1, 2, 0).numpy().clip(0, 1)
            axes[row, 2].imshow(crop_img)
        else:
            axes[row, 2].text(
                0.5, 0.5, "detection failed", ha="center", va="center",
                transform=axes[row, 2].transAxes, fontsize=9,
            )
            axes[row, 2].set_facecolor("#f0f0f0")

        if result.class_probabilities is not None:
            pred_class = CLASSES[result.class_label]
            confidence = result.class_probabilities[result.class_label]
            correct = result.class_label == gt_label
            color = "green" if correct else "red"
            axes[row, 3].bar(CLASSES, result.class_probabilities, color="steelblue")
            axes[row, 3].set_ylim(0, 1)
            axes[row, 3].set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=7)
            axes[row, 3].set_ylabel("Probability", fontsize=8)
            axes[row, 3].set_title(
                f"pred: {pred_class} ({confidence:.2f})", fontsize=9, color=color,
            )
        else:
            axes[row, 3].text(
                0.5, 0.5, "detection failed", ha="center", va="center",
                transform=axes[row, 3].transAxes, fontsize=9,
            )
            axes[row, 3].set_facecolor("#f0f0f0")

        for ax in axes[row]:
            ax.axis("off") if ax != axes[row, 3] else None
        axes[row, 3].set_xticks(range(len(CLASSES)))
        axes[row, 3].set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=7)

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved to {save_path}")
    plt.show()


vis_records = collect_pipeline_results(pipeline, pipeline_dataset, n=CFG["n_visualise"])
visualise_pipeline_outputs(
    vis_records, n=CFG["n_visualise"],
    save_path=Path("/kaggle/working") / "pipeline_outputs.png",
)

## 7. Failure case analysis

In [ ]:
@torch.no_grad()
def analyse_failures(
        pipeline: ISICPipeline,
        dataset: ISICPipelineDataset,
) -> None:
    """
    Categorise failure modes across the full evaluation set.

    Failure categories:
      - Detection failure: Mask R-CNN produced no detection above score_threshold.
      - Segmentation failure: detection succeeded but thresholded Jaccard = 0.0
        (IoU below T=0.65).
      - Classification failure given correct segmentation: Jaccard >= T=0.65 but
        predicted class is wrong.
      - Full success: Jaccard >= T=0.65 and predicted class is correct.
    """
    n_detection_fail = 0
    n_seg_fail = 0
    n_cls_fail = 0
    n_success = 0

    for image, gt_mask, gt_label in torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=2,
    ):
        result = pipeline.predict(image[0])
        gt_mask_np = gt_mask[0].numpy().astype(bool)
        gt_label_int = int(gt_label.item())
        jaccard = pipeline._compute_jaccard(
            result.mask, gt_mask_np, pipeline.jaccard_threshold
        )

        if result.detection_failed:
            n_detection_fail += 1
        elif jaccard == 0.0:
            n_seg_fail += 1
        elif result.class_label != gt_label_int:
            n_cls_fail += 1
        else:
            n_success += 1

    n_total = n_detection_fail + n_seg_fail + n_cls_fail + n_success
    print(f"Failure analysis over {n_total} images:")
    print(
        f"  Detection failure:                        {n_detection_fail:3d} ({100 * n_detection_fail / n_total:.1f}%)")
    print(f"  Segmentation failure (IoU < 0.65):        {n_seg_fail:3d} ({100 * n_seg_fail / n_total:.1f}%)")
    print(f"  Classification failure (correct seg):     {n_cls_fail:3d} ({100 * n_cls_fail / n_total:.1f}%)")
    print(f"  Full success:                             {n_success:3d} ({100 * n_success / n_total:.1f}%)")


analyse_failures(pipeline, pipeline_dataset)